# Le Xin — split model (replaces the merge step)

**Why split, not merge?** Our presentation's pilot used *untuned* Basic Pitch, which wrote too many notes (208 for 156 sung), so chopped notes needed **merging**. The project now uses *tuned* Basic Pitch (onset threshold 0.99). It only starts a new note when it is very sure, so it now writes **too few** notes (7,623 for 8,040 sung) and often runs two sung notes together as one long note.

On the training songs (after the timing fix):
- perfect **merging** could add at most **+0.005** onset F1;
- perfect **splitting** could add **+0.07 to +0.11** onset F1.

So this notebook builds a model that finds where one Basic Pitch note is really two (or more) and **splits** it.

**Input:** Le Xin's timing-corrected notes (out-of-fold by song), so splitting is tested on top of the timing fix.

**Rules:** audio and Basic Pitch notes only as model input (no TextGrid, no reference notes); the answer key is used only for labels and scoring; split by song; test data stays locked.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Open from this notebook folder, or run from the project root.
HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'selected_100.csv').exists())
ROOT = (HERE / '..' / '..').resolve()
sys.path.insert(0, str(HERE))
import correction_utils as cu
DATA = HERE / '..' / '..' / 'basic_pitch_runs' / 'correction_handover'

# The end-to-end scorer needs librosa + mir_eval; fall back to the stand-in if they are missing.
try:
    import librosa, mir_eval.transcription
    print('Scorer: real librosa + mir_eval')
except ImportError:
    sys.path.insert(0, str(ROOT / 'le_xin_work' / 'tools'))
    import shim
    print('Scorer: shim.py stand-in')
import librosa

manifest   = pd.read_csv(DATA / 'training_recording_manifest.csv')
references = pd.read_csv(DATA / 'references_train.csv', dtype={'note_id':str, 'recording_id':str})   # answer key: labels and scoring only
folds      = pd.read_csv(DATA / 'folds.csv').set_index('sample_id')['fold']
raw_notes  = pd.read_csv(DATA / 'train_features.csv', dtype={'note_id':str, 'recording_id':str, 'sample_id':str})
assert raw_notes['split'].eq('train').all() and manifest['split'].eq('train').all()      # test stays locked

# Input to the split step = Le Xin's timing-corrected notes (out-of-fold by song, from the timing notebook).
timed = pd.read_csv(HERE / 'exports' / 'le_xin_timing_rf_audio_pitch_v2_nestedgate.csv', dtype={'note_id':str, 'recording_id':str})
timed = timed.merge(raw_notes[['note_id', 'sample_id']], on='note_id', validate='one_to_one')
assert len(timed) == len(raw_notes)

def f1(table):
    s = cu.evaluate_end_to_end(table[cu.NOTE_COLUMNS], references, manifest)['summary']
    return round(s['song_macro_f1'], 4), round(s['song_macro_offset_f1'], 4)
print('Notes after timing:', len(timed), '| true notes:', len(references), '| onset/offset F1 after timing:', f1(timed))

Scorer: real librosa + mir_eval
Notes after timing: 7623 | true notes: 8040 | onset/offset F1 after timing: (0.3541, 0.2176)


## Step 1: find possible cut points inside each note

For every Basic Pitch note we look *inside* it (never within 60 ms of its edges) for moments where a new note might begin. These come from the audio only:
- the **sung pitch changes** by at least half a semitone (pyin pitch tracker, ~58 ms before vs ~58 ms after);
- a **new attack** (onset strength peak);
- the **voice switches back on** after a short silence.

Each candidate gets clues: how big the pitch change is, how strong the attack is, whether the loudness dips, where in the note it is, and so on.

**Labels (answer key only):** a candidate is a *real* cut if a true note starts within 50 ms of it, inside this Basic Pitch note, and no other Basic Pitch note already starts there.

Pitch tracking takes about 6 minutes the first time; the result is cached in `le_xin_cache/`.

In [2]:
# Step 1: find possible cut points INSIDE each note, from the audio only, and label them with the answer key
SR, HOP = 22050, 256
FRAME = HOP / SR                 # ~11.6 ms per frame
SIDE = 5                         # compare ~58 ms before vs ~58 ms after a frame
EDGE = 0.06                      # never cut closer than 60 ms to a note's start or end
FAR = 12.0                       # 'no pitch found' stand-in (one octave)
CACHE = HERE / 'le_xin_cache' / 'split_candidates_train_v1.csv'

def tracks(path):
    """Per ~11.6 ms frame: sung pitch (MIDI, NaN when silent), voice on/off, onset strength, loudness (dB)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    f0, voiced, vprob = librosa.pyin(y, fmin=65, fmax=1100, sr=SR, hop_length=HOP)
    midi = np.where(voiced, librosa.hz_to_midi(np.nan_to_num(f0, nan=1.0)), np.nan)
    flux = librosa.onset.onset_strength(y=y, sr=SR, hop_length=HOP)
    flux = flux / (np.percentile(flux, 99) + 1e-9)
    db = librosa.amplitude_to_db(librosa.feature.rms(y=y, hop_length=HOP)[0], ref=np.max)
    n = min(len(midi), len(flux), len(db))
    return midi[:n], voiced[:n].astype(bool), vprob[:n], flux[:n], db[:n]

def med(midi, a, b):
    seg = midi[max(a, 0):max(b, 0)]
    seg = seg[~np.isnan(seg)]
    return np.median(seg) if len(seg) >= 2 else np.nan

def candidates_for_note(n, midi, voiced, vprob, flux, db):
    """Frames inside the note where the pitch changes, a new attack starts, or the voice switches on."""
    lo, hi = int(np.ceil((n.start + EDGE) / FRAME)), int((n.end - EDGE) / FRAME)
    hi = min(hi, len(midi) - 1)
    if hi <= lo:
        return []
    frames = np.arange(lo, hi + 1)
    before = np.array([med(midi, i - SIDE, i) for i in frames])
    after  = np.array([med(midi, i, i + SIDE) for i in frames])
    change = np.nan_to_num(np.abs(after - before), nan=0.0)          # semitones
    def peaks(score, floor):
        return {f for k, f in enumerate(frames)
                if score[k] >= floor and score[k] == score[max(0, k - 3):k + 4].max()}
    picked = peaks(change, 0.5) | peaks(flux[frames], 0.25) | {f for f in frames if voiced[f] and not voiced[f - 1]}
    # Keep the strongest frame within any 45 ms cluster
    strength = {f: change[f - lo] + flux[f] for f in picked}
    kept = []
    for f in sorted(picked, key=lambda f: -strength[f]):
        if all(abs(f - g) > 3 for g in kept):
            kept.append(f)
    rows = []
    note_lo, note_hi = int(n.start / FRAME), min(int(n.end / FRAME), len(midi))
    note_db = np.median(db[note_lo:note_hi]) if note_hi > note_lo else db[lo]
    for f in sorted(kept):
        k, t = f - lo, f * FRAME
        b, a = before[k], after[k]
        rows.append(dict(
            note_id=n.note_id, recording_id=n.recording_id, sample_id=n.sample_id, t=t,
            since_start=t - n.start, to_end=n.end - t, rel_pos=(t - n.start) / (n.end - n.start),
            note_dur=n.end - n.start, note_pitch=n.pitch,
            pitch_change=change[k], pitch_change_signed=0.0 if np.isnan(a - b) else a - b,
            before_vs_note=FAR if np.isnan(b) else min(abs(b - n.pitch), FAR),
            after_vs_note=FAR if np.isnan(a) else min(abs(a - n.pitch), FAR),
            flux=flux[max(f - 2, 0):f + 3].max(),
            rise=np.diff(db[max(f - 3, 0):f + 4]).max() if f + 4 - max(f - 3, 0) > 1 else 0.0,
            dip=note_db - db[max(f - 3, 0):f + 4].min(),
            silent_near=float((~voiced[max(f - 3, 0):f + 4]).any()),
            vprob=vprob[f],
            # pitch of the piece before / after this cut, for building the new notes later
            piece_pitch_before=med(midi, max(int(t / FRAME) - 13, note_lo), f),
            piece_pitch_after=med(midi, f, min(f + 13, note_hi))))
    return rows

def true_cut_times(rec_notes, rec_refs):
    """Answer key: a true note starts inside this Basic Pitch note (not near its edges), and no other
    Basic Pitch note already starts there. Returns {note_id: [times]}."""
    starts = rec_notes['start'].to_numpy()
    out = {}
    for n in rec_notes.itertuples():
        inside = rec_refs[(rec_refs.start > n.start + 0.05) & (rec_refs.start < n.end - 0.05)].start.to_numpy()
        inside = [b for b in inside if not (np.abs(starts - b) <= 0.08).any()]
        if inside:
            out[n.note_id] = inside
    return out

def build_candidates(note_table, man, refs, split):
    assert split in ('train', 'val')                                  # test stays locked
    rows, needed, found = [], 0, 0
    for rec in man.itertuples():
        assert rec.split == split and rec.audio_path.startswith(f'data/{split}/')
        rec_notes = note_table[note_table.recording_id == rec.recording_id]
        if rec_notes.empty:
            continue
        curves = tracks(ROOT / rec.audio_path)
        cand = pd.DataFrame([r for n in rec_notes.itertuples() for r in candidates_for_note(n, *curves)])
        truth = true_cut_times(rec_notes, refs[refs.recording_id == rec.recording_id]) if refs is not None else {}
        if not cand.empty:
            cand['label'] = 0
            for nid, times in truth.items():
                c = cand[cand.note_id == nid]
                for b in times:
                    needed += 1
                    if len(c) and np.abs(c.t - b).min() <= 0.05:
                        cand.loc[(c.t - b).abs().idxmin(), 'label'] = 1
                        found += 1
        else:
            needed += sum(len(v) for v in truth.values())
        rows.append(cand)
    return pd.concat(rows, ignore_index=True), needed, found

if CACHE.exists():
    cands = pd.read_csv(CACHE, dtype={'note_id':str, 'recording_id':str, 'sample_id':str})
    print('Loaded cached candidates:', CACHE.name)
else:
    cands, needed, found = build_candidates(timed, manifest, references, 'train')   # ~5-6 minutes (pitch tracking)
    CACHE.parent.mkdir(exist_ok=True)
    cands.to_csv(CACHE, index=False)
    print('Computed and cached candidates:', CACHE.name)

truth_all = {k: v for rid, g in timed.groupby('recording_id')
             for k, v in true_cut_times(g, references[references.recording_id == rid]).items()}
needed = sum(len(v) for v in truth_all.values())
found = int(cands.label.sum())
print(f'Basic Pitch notes that hide a true note start inside them: {len(truth_all)} of {len(timed)}')
print(f'True cut points needed: {needed} | candidate cut points found: {len(cands)} '
      f'({cands.note_id.nunique()} notes have at least one)')
print(f'Candidates on a true cut point (within 50 ms): {found} = {found / needed:.0%} of the needed cuts '
      f'| share of candidates that are real: {cands.label.mean():.1%}')

Loaded cached candidates: split_candidates_train_v1.csv
Basic Pitch notes that hide a true note start inside them: 1593 of 7623
True cut points needed: 1924 | candidate cut points found: 19729 (7009 notes have at least one)
Candidates on a true cut point (within 50 ms): 1217 = 63% of the needed cuts | share of candidates that are real: 6.2%


### How much could splitting at our candidates help?

Before training anything, check the ceiling: if a model picked **exactly** the real candidates, how much would F1 rise? And what happens if we split at every candidate (no model)? Each new piece gets its pitch from the audio (rounded sung pitch); the first piece keeps the original note ID.

In [3]:
# How much could splitting at OUR candidates help, if a model picked exactly the right ones?
def apply_splits(note_table, cuts):
    """COMPLETE replacement note table: split each note at its accepted cut times.
    First piece keeps the original note_id; new pieces get note_id~1, ~2, ...
    Piece pitch: rounded sung pitch from the audio when available, else the original pitch."""
    out, by_note = [], {nid: g.sort_values('t') for nid, g in cuts.groupby('note_id')}
    for n in note_table.itertuples():
        c = by_note.get(n.note_id)
        if c is None or c.empty:
            out.append(dict(note_id=n.note_id, recording_id=n.recording_id, start=n.start, end=n.end, pitch=n.pitch))
            continue
        edges = [n.start, *c.t.tolist(), n.end]
        first = c.piece_pitch_before.iloc[0]
        pitches = [first, *c.piece_pitch_after.tolist()]
        for k in range(len(edges) - 1):
            p = pitches[k]
            out.append(dict(note_id=n.note_id if k == 0 else f'{n.note_id}~{k}', recording_id=n.recording_id,
                            start=edges[k], end=edges[k + 1],
                            pitch=float(np.clip(round(p), 0, 127)) if not np.isnan(p) else n.pitch))
    return pd.DataFrame(out)[cu.NOTE_COLUMNS]

rows = []
for name, table in [('Timing only (no split)',                   timed),
                    ('Split at EVERY candidate (no model)',      apply_splits(timed, cands)),
                    ('Split only at TRUE candidates (ceiling)',  apply_splits(timed, cands[cands.label == 1]))]:
    onset, offset = f1(table)
    rows.append(dict(version=name, notes=len(table), onset_F1=onset, offset_F1=offset))
ceiling = pd.DataFrame(rows).set_index('version')
display(ceiling)

,notes,onset_F1,offset_F1
version,,,
Timing only (no split),7623,0.3541,0.2176
Split at EVERY candidate (no model),27352,0.1829,0.0447
Split only at TRUE candidates (ceiling),8840,0.4194,0.2942


## Step 2: learn which candidates are real cuts

Only about 6% of candidates are real cuts, so a model has to be picky. We compare three ways of choosing, each tested out-of-fold by song (train on 4 song groups, test on the 5th):
1. **Rule:** cut wherever the sung pitch changes by at least X semitones.
2. **Decision tree** and 3. **random forest** on 19 clues per candidate (pitch change, attack strength, loudness dip, voice on/off, position in the note, how the ~150 ms pieces on each side compare with the note's pitch, ...).

The rule's X and each model's probability threshold are chosen **inside the training folds** (nested), by end-to-end onset F1, so the held-out song group never influences its own setting.

This takes about 4–5 minutes.

In [4]:
# Step 2: learn which candidates are real cuts: rule vs decision tree vs random forest, out-of-fold by song
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Clues per candidate (audio + Basic Pitch note only), plus two "compared with the rest of this note" clues
def add_clues(df):
    """Extra clues computed from the candidate table itself (audio + Basic Pitch note only)."""
    df = df.copy()
    df['cands_in_note'] = df.groupby('note_id').t.transform('size')
    df['strength_rank'] = df.assign(s=df.pitch_change + df.flux).groupby('note_id').s.rank(ascending=False)
    # Longer view (~150 ms each side): would the two pieces be different whole notes?
    pb, pa = df.piece_pitch_before, df.piece_pitch_after
    df['piece_change'] = (pa.round() - pb.round()).abs().fillna(FAR)          # semitones between the rounded pieces
    df['piece_after_vs_note'] = (pa - df.note_pitch).abs().fillna(FAR).clip(upper=FAR)
    df['piece_before_vs_note'] = (pb - df.note_pitch).abs().fillna(FAR).clip(upper=FAR)
    return df
cands = add_clues(cands)
FEATS = ['since_start', 'to_end', 'rel_pos', 'note_dur', 'note_pitch', 'pitch_change', 'pitch_change_signed',
         'before_vs_note', 'after_vs_note', 'flux', 'rise', 'dip', 'silent_near', 'vprob', 'cands_in_note', 'strength_rank',
         'piece_change', 'piece_after_vs_note', 'piece_before_vs_note']
Xc, yc = cands[FEATS].to_numpy(float), cands.label.to_numpy()
cand_fold = cands.sample_id.map(folds).to_numpy()
rec_fold = manifest.sample_id.map(folds)
ALL = sorted(np.unique(cand_fold))

def f1_on(fold_set, cut_mask):
    """End-to-end onset/offset F1 on the songs in fold_set after splitting at the chosen candidates."""
    recs = manifest[rec_fold.isin(fold_set)]
    keep = timed.recording_id.isin(recs.recording_id)
    table = apply_splits(timed[keep], cands[cut_mask & np.isin(cand_fold, fold_set)])
    s = cu.evaluate_end_to_end(table, references[references.recording_id.isin(recs.recording_id)], recs)['summary']
    return s['song_macro_f1'], s['song_macro_offset_f1']

def oof_prob(make_model, fold_set):
    """Train on the other folds in fold_set, predict each fold's candidates (probability of a real cut)."""
    p = np.zeros(len(cands))
    for k in fold_set:
        test, train = cand_fold == k, np.isin(cand_fold, [o for o in fold_set if o != k])
        p[test] = make_model().fit(Xc[train], yc[train]).predict_proba(Xc[test])[:, 1]
    return p

THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
def nested(make_model):
    """Outer: each held-out fold. Inner: choose the probability threshold on the other 4 folds only."""
    outer, cut, chosen = oof_prob(make_model, ALL), np.zeros(len(cands), bool), {}
    for k in ALL:
        others = [o for o in ALL if o != k]
        inner = oof_prob(make_model, others)
        th = max(THRESHOLDS, key=lambda t: f1_on(others, inner >= t)[0])
        chosen[int(k)] = th
        cut |= (cand_fold == k) & (outer >= th)
    return cut, chosen, outer

# 1. Rule (no learning): cut wherever the sung pitch changes by at least X semitones; X chosen on the other folds
RULE_X = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]
rule_cut, rule_chosen = np.zeros(len(cands), bool), {}
for k in ALL:
    others = [o for o in ALL if o != k]
    x = max(RULE_X, key=lambda v: f1_on(others, cands.pitch_change.to_numpy() >= v)[0])
    rule_chosen[int(k)] = x
    rule_cut |= (cand_fold == k) & (cands.pitch_change.to_numpy() >= x)

# 2-3. Decision tree and random forest (about 4 minutes in total)
dt_cut, dt_chosen, dt_prob = nested(lambda: DecisionTreeClassifier(min_samples_leaf=40, class_weight='balanced', random_state=0))
rf_cut, rf_chosen, rf_prob = nested(lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                                                    class_weight='balanced_subsample', n_jobs=-1, random_state=0))
print('Chosen per fold | rule (semitones):', rule_chosen, '| tree threshold:', dt_chosen, '| forest threshold:', rf_chosen)

rows = []
for name, cut in [('Timing only (no split)', np.zeros(len(cands), bool)), ('Rule: pitch change', rule_cut),
                  ('Decision tree', dt_cut), ('Random forest', rf_cut), ('Ceiling (perfect picks)', yc == 1)]:
    table = apply_splits(timed, cands[cut])
    onset, offset = f1(table)
    tp = int((cut & (yc == 1)).sum())
    rows.append(dict(model=name, cuts=int(cut.sum()), real_cuts=tp,
                     cut_precision=round(tp / cut.sum(), 3) if cut.sum() else None,
                     share_of_real_found=round(tp / yc.sum(), 3), notes=len(table), onset_F1=onset, offset_F1=offset))
split_results = pd.DataFrame(rows).set_index('model')
display(split_results)

Chosen per fold | rule (semitones): {0: 6.0, 1: 6.0, 2: 6.0, 3: 6.0, 4: 6.0} | tree threshold: {0: 0.95, 1: 0.95, 2: 0.95, 3: 0.95, 4: 0.9} | forest threshold: {0: 0.5, 1: 0.5, 2: 0.5, 3: 0.5, 4: 0.5}


,cuts,real_cuts,cut_precision,share_of_real_found,notes,onset_F1,offset_F1
model,,,,,,,
Timing only (no split),0,0,NaN,0.000,7623,0.3541,0.2176
Rule: pitch change,18,1,0.056,0.001,7641,0.3534,0.2170
Decision tree,121,36,0.298,0.030,7744,0.3545,0.2200
Random forest,359,133,0.370,0.109,7982,0.3585,0.2243
Ceiling (perfect picks),1217,1217,1.000,1.000,8840,0.4194,0.2942


### Step 2b: does Basic Pitch's own onset probability help?

Basic Pitch only starts a note when its onset probability passes 0.99. Inside a long note, a weaker onset peak might still mark a real new note. We re-ran Basic Pitch (the project's ONNX model) on every training recording to get its onset and note probabilities, checked that decoding them with the tuned settings reproduces the frozen predictions exactly (534 of 534 recordings), and added 6 clues per candidate: the strongest onset probability nearby (any pitch, the note's pitch, the pitch sung next), and whether the note activation sags.

The clues are cached in `le_xin_cache/split_bp_onset_train_v1.csv`. To rebuild them you need `basic-pitch` and `onnxruntime` (see `requirements-lock.txt`) and `le_xin_work/tools/bp_onset_features.py`; this notebook only reads the cache.

In [5]:
# Step 2b: add Basic Pitch's own onset probabilities inside the note as clues
BPCACHE = HERE / 'le_xin_cache' / 'split_bp_onset_train_v1.csv'     # made by le_xin_work/tools/bp_onset_features.py
bp = pd.read_csv(BPCACHE, dtype={'note_id':str})
BP_COLS = [c for c in bp.columns if c.startswith('bp_')]
key = lambda d: d.note_id + '@' + d.t.round(4).astype(str)              # match rows by note and cut time
bp = bp.set_index(key(bp))[BP_COLS]
assert bp.index.is_unique and len(bp) == len(cands)
cands[BP_COLS] = bp.loc[key(cands)].to_numpy()
print('Average Basic Pitch onset probability at the candidate:',
      cands.groupby('label').bp_onset_any.mean().round(3).rename({0: 'fake cuts', 1: 'real cuts'}).to_dict())

Xc = cands[FEATS + BP_COLS].to_numpy(float)                          # oof_prob/nested read Xc
rf_bp_cut, rf_bp_chosen, rf_bp_prob = nested(lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                                                             class_weight='balanced_subsample', n_jobs=-1, random_state=0))
print('Forest + Basic Pitch clues, threshold per fold:', rf_bp_chosen)
table = apply_splits(timed, cands[rf_bp_cut])
onset, offset = f1(table)
tp = int((rf_bp_cut & (yc == 1)).sum())
split_results.loc['Random forest + Basic Pitch onset clues'] = dict(
    cuts=int(rf_bp_cut.sum()), real_cuts=tp, cut_precision=round(tp / max(rf_bp_cut.sum(), 1), 3),
    share_of_real_found=round(tp / yc.sum(), 3), notes=len(table), onset_F1=onset, offset_F1=offset)
order = [i for i in split_results.index if i != 'Ceiling (perfect picks)']
split_results = split_results.loc[order + [i for i in split_results.index if i not in order]]   # keep the ceiling row last
display(split_results)

Average Basic Pitch onset probability at the candidate: {'fake cuts': 0.604, 'real cuts': 0.579}
Forest + Basic Pitch clues, threshold per fold: {0: 0.4, 1: 0.5, 2: 0.5, 3: 0.5, 4: 0.5}


,cuts,real_cuts,cut_precision,share_of_real_found,notes,onset_F1,offset_F1
model,,,,,,,
Timing only (no split),0,0,NaN,0.000,7623,0.3541,0.2176
Rule: pitch change,18,1,0.056,0.001,7641,0.3534,0.2170
Decision tree,121,36,0.298,0.030,7744,0.3545,0.2200
Random forest,359,133,0.370,0.109,7982,0.3585,0.2243
Random forest + Basic Pitch onset clues,420,159,0.379,0.131,8043,0.3586,0.2256
Ceiling (perfect picks),1217,1217,1.000,1.000,8840,0.4194,0.2942


## Step 3: final split model, validation songs, export

- **Final model:** the random forest (without the Basic Pitch clues, which did not help) trained on **all** labelled training candidates. Its threshold is the one with the best training out-of-fold onset F1, so the validation songs play no part in any choice.
- **Validation input:** the timing model's corrected validation notes (`le_xin_timing_val_rf_audio_pitch.csv`). Candidates are found the same way (audio only; cached, ~2 minutes the first time). The validation answer key is parsed with the project's own parser and used **only for scoring**.
- **Export:** complete note tables (`note_id, recording_id, start, end, pitch`; a split note keeps its ID for the first piece, new pieces get `note_id~1`, `~2`, ...) plus an action history listing each split note and its cut times. Files are written once; a rerun checks they still match.

In [6]:
# Step 3: final split model -> validation songs -> export (Basic Pitch clues NOT used: they did not help)
sys.path.insert(0, str((HERE / '..').resolve()))
import bp_experiment as bx                                   # project's reference parser (val answer key, scoring only)

# 1. Final model: random forest on ALL labelled training candidates; threshold = best training out-of-fold onset F1
final_threshold = max(THRESHOLDS, key=lambda t: f1_on(ALL, rf_prob >= t)[0])
final_rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                  class_weight='balanced_subsample', n_jobs=-1, random_state=0).fit(cands[FEATS], yc)
print('Final threshold (chosen on training songs only):', final_threshold)

# 2. Validation input = the timing model's corrected validation notes (exported by the timing notebook)
all_manifest = pd.read_csv(DATA / 'recording_manifest.csv')
val_manifest = all_manifest[all_manifest.split == 'val'].reset_index(drop=True)
val_raw = pd.read_csv(DATA / 'val_features.csv', dtype={'note_id':str, 'recording_id':str, 'sample_id':str})
val_timed = pd.read_csv(HERE / 'exports' / 'le_xin_timing_val_rf_audio_pitch.csv', dtype={'note_id':str, 'recording_id':str})
val_timed = val_timed.merge(val_raw[['note_id', 'sample_id']], on='note_id', validate='one_to_one')
assert val_manifest.split.eq('val').all() and len(val_timed) == len(val_raw)            # never test
val_events, _ = bx.references(val_manifest.to_dict('records'))
val_refs = pd.DataFrame([dict(note_id=f'{rid}:ref{i:05d}', recording_id=rid, start=e[0], end=e[1], pitch=e[2])
                         for rid, events in val_events.items() for i, e in enumerate(events)])

VCACHE = HERE / 'le_xin_cache' / 'split_candidates_val_v1.csv'
if VCACHE.exists():
    val_cands = pd.read_csv(VCACHE, dtype={'note_id':str, 'recording_id':str, 'sample_id':str})
else:
    val_cands, _, _ = build_candidates(val_timed, val_manifest, None, 'val')   # no labels: answer key not used here
    val_cands.to_csv(VCACHE, index=False)                                      # ~2 minutes (pitch tracking)
val_cands = add_clues(val_cands)
val_cut = final_rf.predict_proba(val_cands[FEATS])[:, 1] >= final_threshold

def score(table, refs, man):
    out = cu.evaluate_end_to_end(table[cu.NOTE_COLUMNS], refs, man)
    assert out['summary']['recordings'] == len(man)
    return out
val_before = score(val_timed, val_refs, val_manifest)
val_split = apply_splits(val_timed, val_cands[val_cut])
val_after = score(val_split, val_refs, val_manifest)
before_song = {s['sample_id']: s['f1'] for s in val_before['songs']}
better = sum(s['f1'] > before_song[s['sample_id']] + 1e-12 for s in val_after['songs'])
worse = sum(s['f1'] < before_song[s['sample_id']] - 1e-12 for s in val_after['songs'])
train_split = apply_splits(timed, cands[rf_cut])                               # training: out-of-fold cuts (step 2)
tr_before, tr_after = f1(timed), f1(train_split)
val_table = pd.DataFrame([
    dict(songs='Training (out-of-fold)', step='Timing only',    notes=len(timed),       onset_F1=tr_before[0], offset_F1=tr_before[1]),
    dict(songs='Training (out-of-fold)', step='Timing + split', notes=len(train_split), onset_F1=tr_after[0],  offset_F1=tr_after[1]),
    dict(songs='Validation (20 songs)',  step='Timing only',    notes=len(val_timed),
         onset_F1=round(val_before['summary']['song_macro_f1'], 4), offset_F1=round(val_before['summary']['song_macro_offset_f1'], 4)),
    dict(songs='Validation (20 songs)',  step='Timing + split', notes=len(val_split),
         onset_F1=round(val_after['summary']['song_macro_f1'], 4), offset_F1=round(val_after['summary']['song_macro_offset_f1'], 4)),
]).set_index(['songs', 'step'])
display(val_table)
print(f'Validation: {int(val_cut.sum())} cuts in {val_cands[val_cut].note_id.nunique()} notes | '
      f'songs better: {better}, worse: {worse}, unchanged: {len(val_after["songs"]) - better - worse} (onset F1, vs timing only)')

# 3. Export complete note tables + action histories (write once; on a rerun, check the saved files still match)
def split_history(cut_table):
    rows = []
    for nid, g in cut_table.sort_values('t').groupby('note_id'):
        rows.append(dict(action='split', note_id=nid, recording_id=g.recording_id.iloc[0],
                         cut_times=';'.join(f'{t:.4f}' for t in g.t),
                         new_note_ids=';'.join(f'{nid}~{k}' for k in range(1, len(g) + 1))))
    return pd.DataFrame(rows, columns=['action', 'note_id', 'recording_id', 'cut_times', 'new_note_ids'])

def save_once(table, path, man=None):
    if path.exists():
        saved = pd.read_csv(path, dtype={'note_id':str, 'recording_id':str})
        pd.testing.assert_frame_equal(saved, table.reset_index(drop=True), check_dtype=False)
        print('Already saved, unchanged:', path.name)
    elif man is not None:
        cu.export_corrections(table, man, path)
        print('Saved:', path.name)
    else:
        with path.open('x', encoding='utf-8', newline='') as handle:
            table.to_csv(handle, index=False)
        print('Saved:', path.name)

EXPORTS = HERE / 'exports'
save_once(train_split, EXPORTS / 'le_xin_split_train_oof_v1.csv', manifest)
save_once(split_history(cands[rf_cut]), EXPORTS / 'le_xin_split_train_oof_v1_actions.csv')
save_once(val_split, EXPORTS / 'le_xin_split_val_v1.csv', val_manifest)
save_once(split_history(val_cands[val_cut]), EXPORTS / 'le_xin_split_val_v1_actions.csv')

Final threshold (chosen on training songs only): 0.5


notes  onset_F1  offset_F1
songs                  step                                      
Training (out-of-fold) Timing only      7623    0.3541     0.2176
                       Timing + split   7982    0.3585     0.2243
Validation (20 songs)  Timing only      2329    0.3755     0.2360
                       Timing + split   2437    0.3796     0.2427

Validation: 108 cuts in 91 notes | songs better: 14, worse: 5, unchanged: 1 (onset F1, vs timing only)
Saved: le_xin_split_train_oof_v1.csv
Saved: le_xin_split_train_oof_v1_actions.csv
Saved: le_xin_split_val_v1.csv
Saved: le_xin_split_val_v1_actions.csv
